In [2]:
import uuid
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.types import *

StatementMeta(, 86e209c4-97cb-4208-bebb-fbcbca8ebd40, 4, Finished, Available, Finished, False)

In [1]:
def log_start(spark, run_id, file_name, entity, load_type):

    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("entity", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
    ])

    data = [{
        "run_id": str(run_id),
        "file_name": str(file_name),
        "entity": str(entity),
        "load_type": str(load_type),

        "rows_read": 0,
        "rows_inserted": 0,
        "rows_updated": 0,

        "status": "RUNNING",
        "error_message": "",

        "start_time": datetime.utcnow(),
        "end_time": None
    }]

    df = spark.createDataFrame(data, schema=schema)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable("audit.pipeline_log")
    )

    return data[0]

StatementMeta(, 86e209c4-97cb-4208-bebb-fbcbca8ebd40, 3, Finished, Available, Finished, False)

In [3]:
def log_end(
    spark,
    ctx,
    status,
    rows_read=0,
    rows_inserted=0,
    rows_updated=0,
    error_message=""
):

    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("end_time", TimestampType(), True),
    ])

    update_data = [{
        "run_id": str(ctx["run_id"]),
        "file_name": str(ctx["file_name"]),
        "status": str(status),

        "rows_read": int(rows_read),
        "rows_inserted": int(rows_inserted),
        "rows_updated": int(rows_updated),

        "error_message": str(error_message) if error_message else "",

        "end_time": datetime.utcnow()
    }]

    df_update = spark.createDataFrame(update_data, schema=schema)

    (
        DeltaTable.forName(spark, "audit.pipeline_log")
        .alias("t")
        .merge(
            df_update.alias("s"),
            "t.run_id = s.run_id AND t.file_name = s.file_name"
        )
        .whenMatchedUpdate(set={
            "status": "s.status",
            "rows_read": "s.rows_read",
            "rows_inserted": "s.rows_inserted",
            "rows_updated": "s.rows_updated",
            "error_message": "s.error_message",
            "end_time": "s.end_time",
        })
        .execute()
    )

StatementMeta(, 28d87509-beff-4904-b5ae-a2c2b8399909, 5, Finished, Available, Finished, False)